### **Vision Transformer Architecture**

In [ ]:
#import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os #check for filepath
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
class PatchEmbedding(nn.Module):
  def __init__(self, patch_size=16, img_size=224, in_channels=3, embedding=768): #set as default
    super().__init__()
    self.n_patches = (img_size // patch_size) ** 2 # 224/16=14

    self.proj = nn.Conv2d(in_channels, out_channels = embedding, kernel_size = patch_size, stride=patch_size)
    #slides over patches and dot product with weights (randomly initialized at first)

  def forward(self, x):
    x = self.proj(x) # (B, 3, 224, 224) -> (B, 768, 14, 14)
    x = x.flatten(2)
    x = x.transpose(1,2) # (B, 196, 768) 196 tokens with 768 features (vector)
    return x

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, n_patches=196, embedding=768):
        super().__init__()

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embedding)) #classification token starts at zero, all tokens are of same dimension
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embedding)) #position of token in learned during training

    def forward(self, x):
        B = x.shape[0]

        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, 768) each image gets a cls token
        x = torch.cat([cls, x], dim=1) # concatenate the cls token with 196 patches (B, 197, 768)
        x = x + self.pos_embed  #each token now unique

        return x  # (B, 197, 768)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding=768, n_heads=12):
        super().__init__()

        self.n_heads = n_heads # 12 independent attention computations

        # 768 / 12 = 64 features
        self.head_dim = embedding // n_heads

        self.queries = nn.Linear(embedding, embedding) #n_inputs = n_outputs
        self.keys = nn.Linear(embedding, embedding)
        self.values = nn.Linear(embedding, embedding)

        self.out_proj = nn.Linear(embedding, embedding)

    def forward(self, x):
        batch, n_tokens, embed_dim = x.shape  # batch, number of tokens (197), embedding dim (768)

        # project every token into queries, keys, values
        Q = self.queries(x)  # (B, 197, 768)
        K = self.keys(x)
        V = self.values(x)

        # split 768 features across 12 heads
        # each head sees 64 features per token
        Q = Q.view(batch, n_tokens, self.n_heads, self.head_dim).transpose(1, 2)  # (B, 12, 197, 64)
        K = K.view(batch, n_tokens, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch, n_tokens, self.n_heads, self.head_dim).transpose(1, 2)

        # compare every query to every key — produces attention score
        # Attention(Q, K, V) = softmax( QK^T / sqrt(d_k) ) * V
        scale = self.head_dim ** 0.5
        scores = (Q @ K.transpose(-2, -1)) / scale  # (B, 12, 197, 197)

        # softmax turns scores into probabilities — all add up to 1
        # tells each token how much to focus on every other token
        weights = torch.softmax(scores, dim=-1)  # (B, 12, 197, 197)

        # use weights to take a weighted sum of values
        attention_output = weights @ V  # (B, 12, 197, 64)

        # merge all 12 heads back together
        attention_output = attention_output.transpose(1, 2).reshape(batch, n_tokens, embed_dim)  # (B, 197, 768)

        # final linear projection
        x = self.out_proj(attention_output)  # (B, 197, 768)

        return x

In [ ]:
class MLP(nn.Module):
    def __init__(self, embedding=768, mlp_ratio=4):
        super().__init__()

        # hidden layer is 4x bigger than the embedding dim
        # 768 * 4 = 3072
        hidden = embedding * mlp_ratio

        self.layer1 = nn.Linear(embedding, hidden)   # expand:  768  → 3072
        self.act_func  = nn.GELU()
        self.layer2  = nn.Linear(hidden, embedding)    # compress: 3072 → 768

    def forward(self, x):
        x = self.layer1(x)   # expand
        x = self.act_func(x)   # add non-linearity
        x = self.layer2(x)   # compress back down
        return x

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embedding=768, n_heads=12, mlp_ratio=4, dropout=0.1): # Added dropout parameter
        super().__init__()

        # normalisation before each sub layer mean 0 std dev 1
        self.norm1 = nn.LayerNorm(embedding)
        self.norm2 = nn.LayerNorm(embedding)

        self.attention = MultiHeadAttention(embedding, n_heads)
        self.mlp       = MLP(embedding, mlp_ratio)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x):
        # attention sub-layer
        # normalise first, run attention, then add the input back (residual)
        # residual means: keep the original x and just add the change on top
        x = x + self.dropout(self.attention(self.norm1(x))) # Applied dropout after attention

        # MLP sub-layer
        # same pattern — normalise, run MLP, add residual
        x = x + self.dropout(self.mlp(self.norm2(x))) # Applied dropout after MLP

        return x  # shape unchanged: (B, 197, 768)

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embedding=256, n_heads=6,
                 n_layers=6, mlp_ratio=4, n_classes=10, dropout=0.1): # Added dropout parameter
        super().__init__()

        n_patches = (img_size // patch_size) ** 2  # 196

        # block 1 — image into patch tokens
        self.patch_embedding = PatchEmbedding(patch_size, img_size, in_channels, embedding)

        # block 2 — add CLS token and position information
        self.positional_encoding = PositionalEncoding(n_patches, embedding)

        # block 3+4+5 — stack of transformer blocks
        # each block runs attention then MLP
        self.transformer_blocks = nn.Sequential(*[
            TransformerBlock(embedding, n_heads, mlp_ratio, dropout) # Passed dropout to TransformerBlock
            for _ in range(n_layers)
        ])

        # final layer norm before classification
        self.norm = nn.LayerNorm(embedding)

        # classification head — takes CLS token and produces class scores
        self.classifier = nn.Linear(embedding, n_classes)

    def forward(self, x):

        # step 1 — cut image into patches and project each to embedding dim
        x = self.patch_embedding(x)        # (B, 3, 224, 224) → (B, 196, embedding)

        # step 2 — prepend CLS token and add position information
        x = self.positional_encoding(x)    # (B, 196, embedding) → (B, 197, embedding)

        # step 3 — pass through transformer blocks
        x = self.transformer_blocks(x)     # (B, 197, embedding)

        # step 4 — normalise
        x = self.norm(x)

        # step 5 — pull out just the CLS token (position 0)
        cls_output = x[:, 0]

        # step 6 — classify using the CLS summary
        out = self.classifier(cls_output)

        return out

### **Training block**


In [ ]:
import os
import pandas as pd
from pathlib import Path

data_path = Path(r"C:\Users\User\OneDrive\Desktop\data")

# Load CSVs
train_df = pd.read_csv(data_path / "train.csv")
test_df  = pd.read_csv(data_path / "test.csv")

# Build local image paths
def fix_path(raw_path):
    clean = str(raw_path).replace('\\', '/')
    parts = clean.split('/')
    class_folder = parts[-2]
    filename = parts[-1]
    return str(data_path / "food-101" / "images" / class_folder / filename)

train_df['image_path'] = train_df['image_path'].apply(fix_path)
test_df['image_path']  = test_df['image_path'].apply(fix_path)

# Build class mappings
classes  = sorted(train_df['label_name'].unique())
cls2idx  = {c: i for i, c in enumerate(classes)}
idx2cls  = {i: c for c, i in cls2idx.items()}
train_df['label'] = train_df['label_name'].map(cls2idx)

N_CLASSES = len(classes)
print(f'{N_CLASSES} classes: {classes}')
print(f'Training images: {len(train_df)} | Test images: {len(test_df)}')

# Verify path looks correct
print(train_df['image_path'].iloc[0])

In [ ]:
# Transforms: resize and normalise every image before feeding it to the model
from torch.utils.data import Dataset  # Import Dataset

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),              # no random augmentation for val/test
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# load images and labels from dataframe, apply transforms
class FoodDataset(Dataset):
    def __init__(self, df, transform, has_labels=True):
        self.paths      = df['image_path'].tolist()
        self.labels     = df['label'].tolist() if has_labels else None
        self.transform  = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert('RGB')
        image = self.transform(image)
        if self.labels is not None:
            return image, self.labels[idx]
        return image

In [ ]:
print(os.listdir(data_path / "food-101"/ "images"))

In [ ]:
import os
from pathlib import Path

# Check if the first image actually exists
first_path = train_df['image_path'].iloc[0]
print(f'Path: {first_path}')
print(f'File exists: {os.path.exists(first_path)}')

# Show what files are actually in that folder
path_obj = Path(first_path)
class_folder = path_obj.parent.name
folder_path = path_obj.parent
print(f'\nFirst 5 files in {class_folder}:')
print(os.listdir(folder_path)[:5])

In [ ]:
from itertools import product
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
import numpy as np

param_grid = {
    'lr'          : [1e-3, 3e-4],   # learning rate — how big each update step is
    'weight_decay': [1e-4, 1e-3],   # regularisation — prevents the model memorising training data
    'dropout'     : [0.0,  0.1],    # randomly switches off neurons during training to prevent overfitting
}

CV_FOLDS  = 3   
CV_EPOCHS = 5   

results      = []                          # we will store each config's result here
combos       = list(product(*param_grid.values()))  # all 8 combinations
keys         = list(param_grid.keys())              # ['lr', 'weight_decay', 'dropout']
labels_array = train_df['label'].values             # needed by StratifiedKFold
train_losses   = []
val_accuracies = []

print(f'{len(combos)} configs × {CV_FOLDS} folds = {len(combos) * CV_FOLDS} total runs\n')

for combo in combos:

    # Turn the combination into a dictionary e.g. {'lr': 0.001, 'weight_decay': 0.0001, 'dropout': 0.0}
    params    = dict(zip(keys, combo))
    fold_accs = []   # store accuracy from each fold for this config

    # StratifiedKFold splits the data into CV_FOLDS parts
    # "Stratified" means each fold has a balanced mix of all 10 classes
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels_array)), labels_array)):

        # Use the indices to get the actual rows from the dataframe
        train_fold = train_df.iloc[train_idx].reset_index(drop=True)
        val_fold   = train_df.iloc[val_idx].reset_index(drop=True)

        train_loader = DataLoader(FoodDataset(train_fold, train_transform), batch_size=16, shuffle=True,  num_workers=0)
        val_loader   = DataLoader(FoodDataset(val_fold,   val_transform),   batch_size=16, shuffle=False, num_workers=0)

        # Build a fresh model for each fold using the current config's dropout value
        model     = VisionTransformer(n_classes=N_CLASSES, embedding=256, n_heads=6, n_layers=6, dropout=params['dropout']).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
        criterion = nn.CrossEntropyLoss()

        # Train for CV_EPOCHS epochs — just enough to see which config learns fastest
        for epoch in range(CV_EPOCHS):
            model.train()
            for batch_idx, (images, labels) in enumerate(train_loader):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                optimizer.step()
                if batch_idx % 50 == 0:
                    print(f"    Epoch {epoch+1}/{CV_EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

        # After training, check how accurate the model is on the validation fold
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():                      # no gradients needed during evaluation
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                correct += (model(images).argmax(1) == labels).sum().item()
                total   += labels.size(0)

        fold_acc = correct / total * 100
        fold_accs.append(fold_acc)
        print(f"  Config {params} | Fold {fold+1}/{CV_FOLDS} | Acc: {fold_acc:.2f}%")

        # Delete the model and clear GPU memory before the next fold
        # Without this, we would run out of GPU memory after a few folds
        del model
        torch.cuda.empty_cache()

    # Average the 3 fold accuracies — this is our score for this config
    mean_acc = np.mean(fold_accs)
    results.append({**params, 'val_acc': round(mean_acc, 2)})
    print(f"→ Average accuracy for this config: {mean_acc:.2f}%\n")

# ── Results ───────────────────────────────────────────────────────────────────
# Sort configs from best to worst and save to CSV for the report
results_df = pd.DataFrame(results).sort_values('val_acc', ascending=False)
results_df.to_csv('grid_search_results.csv', index=False)

print('── Grid Search Results (best to worst) ──────────────')
print(results_df.to_string(index=False))
print(f'\nBest config: lr={results_df.iloc[0].lr}  weight_decay={results_df.iloc[0].weight_decay}  dropout={results_df.iloc[0].dropout}')
print(f'Best CV accuracy: {results_df.iloc[0].val_acc}%')

In [ ]:
best = results_df.iloc[0]
print(f'Using best config → lr={best.lr}  wd={best.weight_decay}  dropout={best.dropout}')
print(f'Grid search CV accuracy: {best.val_acc}%\n')

EPOCHS = 100
WARMUP = 10

# 80/20 split — val_split is your "test" set for evaluation
train_split, val_split = train_test_split(
    train_df, test_size=0.2, stratify=train_df['label'], random_state=42
)
print(f'Train: {len(train_split)} | Test: {len(val_split)}')

train_loader = DataLoader(FoodDataset(train_split, train_transform), batch_size=16, shuffle=True,  num_workers=0)
test_loader  = DataLoader(FoodDataset(val_split,   val_transform),   batch_size=16, shuffle=False, num_workers=0)
sub_loader   = DataLoader(FoodDataset(test_df,     val_transform,    has_labels=False), batch_size=16, shuffle=False, num_workers=0)

model     = VisionTransformer(n_classes=N_CLASSES, embedding=256, n_heads=6, n_layers=6, dropout=float(best.dropout)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=float(best.lr), weight_decay=float(best.weight_decay))
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def get_lr(epoch):
    if epoch < WARMUP:
        return epoch / WARMUP
    progress = (epoch - WARMUP) / (EPOCHS - WARMUP)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler    = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)
best_acc     = 0.0
train_losses = []   # training loss per epoch
test_losses  = []   # test loss per epoch
train_accs   = []   # training accuracy per epoch
test_accs    = []   # test accuracy per epoch

for epoch in range(1, EPOCHS + 1):

    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    scheduler.step()
    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc  = correct / total * 100
    train_losses.append(avg_train_loss)
    train_accs.append(avg_train_acc)

    # ── Test (validation split used as test) ──────────────────────────────────
    model.eval()
    test_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            test_loss += loss.item()
            correct   += (outputs.argmax(1) == labels).sum().item()
            total     += labels.size(0)

    avg_test_loss = test_loss / len(test_loader)
    avg_test_acc  = correct / total * 100
    test_losses.append(avg_test_loss)
    test_accs.append(avg_test_acc)

    current_lr = optimizer.param_groups[0]['lr']

    if avg_test_acc > best_acc:
        best_acc = avg_test_acc
        torch.save(model.state_dict(), 'best_vit.pth')
        flag = ' ← saved'
    else:
        flag = ''

    print(f'Epoch {epoch:02d}/{EPOCHS}  |  LR: {current_lr:.6f}  |  '
          f'Train Loss: {avg_train_loss:.4f}  Train Acc: {avg_train_acc:.2f}%  |  '
          f'Test Loss: {avg_test_loss:.4f}  Test Acc: {avg_test_acc:.2f}%{flag}')

print(f'\nBest test accuracy: {best_acc:.2f}%')

In [ ]:
epochs = range(1, len(train_losses) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot — both train and test
ax1.plot(epochs, train_losses, color='blue',   label='Train loss', linewidth=2)
ax1.plot(epochs, test_losses,  color='orange', label='Test loss',  linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Train vs Test Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot — both train and test
ax2.plot(epochs, train_accs, color='blue',   label='Train accuracy', linewidth=2)
ax2.plot(epochs, test_accs,  color='orange', label='Test accuracy',  linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Train vs Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('ViT Training Progress — Food-101', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('metrics.png')
plt.show()

print(f'Final Train Acc: {train_accs[-1]:.2f}%  |  Best Test Acc: {best_acc:.2f}%')
print(f'Final Train Loss: {train_losses[-1]:.4f}  |  Final Test Loss: {test_losses[-1]:.4f}')

In [ ]:
model.load_state_dict(torch.load('best_vit.pth', map_location=DEVICE))
model.eval()

# Predict on the test set and store the predicted class indices
all_preds = []
with torch.no_grad():                        
    for images in sub_loader:              
        images  = images.to(DEVICE)
        outputs = model(images)              
        preds   = outputs.argmax(dim=1)     
        all_preds.extend(preds.cpu().tolist())

# Convert predicted numbers back to class name strings
pred_labels = [idx2cls[p] for p in all_preds]

# Build the submission file matching Kaggle's expected format
sample_sub          = pd.read_csv('/content/gdrive/MyDrive/data/sample_submission.csv')
submission          = sample_sub.copy()
submission['label'] = pred_labels

# Save it
submission.to_csv('submission.csv', index=False)

print(f'Saved {len(submission)} predictions to submission.csv')
print(submission.head(10))
print(f'\nPrediction counts:')
print(submission['label'].value_counts())